# Fontainebleau benchmark notebook

This notebook stays close to `examples/fontainebleau/run_fontainebleau.py`.
Instead of re-implementing the workflow step by step, it launches the same
`scripts/launcher.py` entry point from a notebook-friendly environment.

The only deliberate difference is that we do not call `--set-results-dir`
interactively. The notebook sets `PYAGE_RESULTS_DIR` for the current session
and then runs the launcher with `--inline`.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "pyproject.toml").exists() and (parent / "pyage").exists():
        ROOT = parent
        break

default_results_root = Path.home() / "results" / "PyAge"
RESULTS_ROOT = Path(os.environ.get("PYAGE_RESULTS_DIR", str(default_results_root)))
os.environ["PYAGE_RESULTS_DIR"] = str(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXAMPLE_DIR = ROOT / "examples" / "fontainebleau"
PARAMS_PATH = EXAMPLE_DIR / "exemple_fontainebleau.yaml"
RUNNER_PATH = EXAMPLE_DIR / "run_fontainebleau.py"

print("CWD:", Path.cwd())
print("ROOT:", ROOT)
print("Example dir:", EXAMPLE_DIR)
print("Params:", PARAMS_PATH)
print("Runner:", RUNNER_PATH)
print("Results root:", RESULTS_ROOT)


In [ ]:
%matplotlib inline


## Reference files

The notebook keeps the current Fontainebleau wrapper and YAML configuration
visible, so it is easy to compare the interactive run with the script-based
entry point.


In [ ]:
import yaml

print("run_fontainebleau.py")
print("=" * 80)
print(RUNNER_PATH.read_text(encoding="utf-8"))

with PARAMS_PATH.open("r", encoding="utf-8") as handle:
    base_config = yaml.safe_load(handle) or {}

print("\nBase YAML")
print("=" * 80)
print(yaml.safe_dump(base_config, sort_keys=False))


## Optional overrides

Edit `overrides` if you want a lighter or alternative benchmark while still
using the same launcher and YAML schema.


In [ ]:
import copy
import tempfile

def deep_update(base: dict, updates: dict) -> dict:
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base

overrides = {
    # "dataset": {"name": "fontainebleau_IMR"},
    # "lpm": {"model_name": "dirac_double"},
    # "run": {
    #     "reachable_concentrations": True,
    #     "objective_function": True,
    #     "calibration_metropolis_hastings": True,
    #     "calibration_simplex": True,
    # },
    # "reachable_concentrations": {"nmodels": 2000},
    # "objective_function": {"nmodels": 4000},
    # "calibration_metropolis_hastings": {"nstep": 1000},
    # "calibration_simplex": {"init_multiples_n": 3, "fuq_n": 20},
}

effective_config = deep_update(copy.deepcopy(base_config), overrides)
if overrides:
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False, encoding="utf-8")
    with tmp:
        yaml.safe_dump(effective_config, tmp, sort_keys=False)
    EFFECTIVE_PARAMS_PATH = Path(tmp.name)
else:
    EFFECTIVE_PARAMS_PATH = PARAMS_PATH

print("Effective params:", EFFECTIVE_PARAMS_PATH)
print(yaml.safe_dump(effective_config, sort_keys=False))


## Run benchmark

This cell mirrors the script launcher path: it executes `scripts/launcher.py`
through `runpy` with notebook-friendly CLI arguments and measures wall time.


In [ ]:
import runpy
from time import perf_counter

import pyage.global_parameters as gp
from scripts.common.launcher_params import load_params
from scripts.common.launcher_paths import results_directory

resolved_params = load_params(ROOT, EFFECTIVE_PARAMS_PATH)
EXPECTED_RESULTS_DIR = Path(results_directory(gp, resolved_params.dataset_name))
script_path = ROOT / "scripts" / "launcher.py"
saved_argv = sys.argv[:]

print("Dataset:", resolved_params.dataset_name)
print("LPM model:", resolved_params.lpm_model_name)
print("Expected results directory:", EXPECTED_RESULTS_DIR)

start = perf_counter()
try:
    sys.argv = [str(script_path), str(EFFECTIVE_PARAMS_PATH), "--inline"]
    runpy.run_path(str(script_path), run_name="__main__")
finally:
    sys.argv = saved_argv

elapsed_s = perf_counter() - start
print(f"Elapsed wall time: {elapsed_s:.2f} s")


## Output check

Quick listing of the most recent files written to the expected results
directory.


In [ ]:
all_files = [path for path in EXPECTED_RESULTS_DIR.rglob("*") if path.is_file()]
recent_files = sorted(all_files, key=lambda path: path.stat().st_mtime, reverse=True)

print("Results directory:", EXPECTED_RESULTS_DIR)
print("File count:", len(all_files))
print("Most recent files:")
for path in recent_files[:20]:
    print(path.relative_to(EXPECTED_RESULTS_DIR))
